In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import cross_val_score
import logging


# --- CONFIGURATION ---
TRAIN_PATH = "/kaggle/input/competitions/industry-5-0-scalable-kinematic-action-recognition/train.csv"
TEST_PATH = "/kaggle/input/competitions/industry-5-0-scalable-kinematic-action-recognition/test.csv"
OUTPUT_PATH = "submission6.csv"

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


def log_phase(title):
    """Logs a formatted phase header to visually separate pipeline stages."""
    logging.info(f"\n{'=' * 60}\n{title}\n{'=' * 60}")


def add_kinematic_features(df, cols):
    """
    Extracts spatial and kinematic features from raw sensor coordinates.

    Engineered features:
        - Rotation/translation aggregate stats (mean, std, max)
        - Left/right body asymmetry
        - Per-axis wrist-elbow and bilateral wrist differences
        - L2 Euclidean norms for key joints (Wrist, Elbow, Humerus)
        - Extended joint pair differences (Elbow-Humerus, Wrist-Humerus)
        - Elbow joint angle via cosine similarity of adjacent bone vectors
        - Bilateral joint distances (L2 norm between left/right joint pairs)
        - Per-segment rotation norms (magnitude of rotation vectors)

    Args:
        df (pd.DataFrame): Raw input dataframe (train or test).
        cols (list[str]): Feature column names to use.

    Returns:
        pd.DataFrame: Augmented dataframe with engineered features appended.
    """
    X = df[cols].fillna(0).copy()
    axes = ['Tx', 'Ty', 'Tz']

    # 1. Rotation Statistics — captures overall rotational magnitude and spread.
    rot_cols = [c for c in cols if c.endswith(('_Rx', '_Ry', '_Rz'))]
    if rot_cols:
        X['rot_mean'] = X[rot_cols].mean(axis=1)
        X['rot_std'] = X[rot_cols].std(axis=1)
        X['rot_max'] = X[rot_cols].max(axis=1)

    # 2. Translation Statistics — captures overall spatial displacement.
    trans_cols = [c for c in cols if c.endswith(('_Tx', '_Ty', '_Tz'))]
    if trans_cols:
        X['trans_mean'] = X[trans_cols].mean(axis=1)
        X['trans_std'] = X[trans_cols].std(axis=1)

    # 3. Left/Right Asymmetry — actions like reaching differ strongly by side.
    left_cols = [c for c in cols if c.startswith('L_')]
    right_cols = [c for c in cols if c.startswith('R_')]
    if left_cols and right_cols:
        X['lr_asymmetry_mean'] = X[left_cols].mean(axis=1) - X[right_cols].mean(axis=1)

    # 4. Spatial Joint Differences — relative displacement between joint pairs.
    #    Extended pairs: Wrist-Elbow, Elbow-Humerus, Wrist-Humerus.
    #    Together they encode full arm chain posture per side and axis.
    joint_pairs = [
        ('Wrist', 'Elbow'),
        ('Elbow', 'Humerus'),
        ('Wrist', 'Humerus'),
    ]
    for side in ['L', 'R']:
        for j1, j2 in joint_pairs:
            for axis in axes:
                c1 = f'{side}_{j1}_{axis}'
                c2 = f'{side}_{j2}_{axis}'
                if c1 in cols and c2 in cols:
                    X[f'{side}_{j1}_{j2}_{axis}'] = X[c1] - X[c2]

    #    Bilateral wrist differences encode bimanual coordination.
    for axis in axes:
        lw, rw = f'L_Wrist_{axis}', f'R_Wrist_{axis}'
        if lw in cols and rw in cols:
            X[f'wrist_diff_{axis}'] = X[lw] - X[rw]

    # 5. L2 Norms — Euclidean distance from origin for key joints,
    #    giving a single scalar for overall joint reach/extension.
    for joint in ['Wrist', 'Elbow', 'Humerus']:
        for side in ['L', 'R']:
            joint_axes = [f'{side}_{joint}_{axis}' for axis in axes]
            if all(c in cols for c in joint_axes):
                X[f'{side}_{joint}_norm'] = np.sqrt(sum(X[c]**2 for c in joint_axes))

    # 6. Elbow Joint Angles — cosine similarity between the upper-arm vector
    #    (Humerus→Elbow) and forearm vector (Wrist→Elbow).
    #    Captures flexion/extension state critical for action discrimination.
    for side in ['L', 'R']:
        w = [f'{side}_Wrist_{ax}' for ax in axes]
        e = [f'{side}_Elbow_{ax}' for ax in axes]
        h = [f'{side}_Humerus_{ax}' for ax in axes]
        if all(c in cols for c in w + e + h):
            e_arr = np.array([X[c].values for c in e])
            vec_we = np.array([X[c].values for c in w]) - e_arr   # forearm vector
            vec_he = np.array([X[c].values for c in h]) - e_arr   # upper-arm vector
            dot    = np.sum(vec_we * vec_he, axis=0)
            norm   = (np.linalg.norm(vec_we, axis=0) * np.linalg.norm(vec_he, axis=0)) + 1e-9
            X[f'{side}_elbow_angle'] = dot / norm

    # 7. Bilateral Joint Distances — L2 distance between left and right
    #    counterpart joints, encoding symmetry and cross-body reach.
    for joint in ['Wrist', 'Elbow', 'Humerus']:
        l_axes = [f'L_{joint}_{ax}' for ax in axes]
        r_axes = [f'R_{joint}_{ax}' for ax in axes]
        if all(c in cols for c in l_axes + r_axes):
            diff = np.array([X[f'L_{joint}_{ax}'].values - X[f'R_{joint}_{ax}'].values
                             for ax in axes])
            X[f'bilateral_{joint}_dist'] = np.linalg.norm(diff, axis=0)

    # 8. Per-Segment Rotation Norms — magnitude of each segment's rotation vector,
    #    summarising total angular displacement in a single scalar per segment.
    segments = set(
        c.rsplit('_', 1)[0]
        for c in cols
        if c.endswith(('_Rx', '_Ry', '_Rz'))
    )
    for seg in segments:
        r_axes = [f'{seg}_Rx', f'{seg}_Ry', f'{seg}_Rz']
        if all(c in cols for c in r_axes):
            X[f'{seg}_rot_norm'] = np.sqrt(sum(X[c]**2 for c in r_axes))

    return X


def load_data():
    """
    Loads train and test CSVs and identifies feature columns.

    Returns:
        tuple: (train_df, test_df, feature_cols)
            - train_df: Full training dataframe including target.
            - test_df: Test dataframe without target.
            - feature_cols: List of input feature column names.
    """
    logging.info("Loading datasets from CSV...")
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)

    # Exclude metadata and target columns — keep only sensor readings.
    feature_cols = [col for col in train_df.columns if col not in ['Id', 'ActivityClass']]
    return train_df, test_df, feature_cols


def build_model():
    """
    Constructs a soft-voting ensemble of LightGBM and Random Forest.

    Ensemble rationale:
        - LightGBM: High accuracy on tabular data with fast training.
        - Random Forest: Strong regularization and variance reduction.
        - Soft voting: Averages class probabilities, rewarding confident predictions.

    Returns:
        VotingClassifier: Untrained ensemble model ready for fitting.
    """
    logging.info("Building hybrid ensemble model (LightGBM + Random Forest)...")

    lgb_model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=5,   # Prevents overfitting on small leaf nodes.
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    )

    return VotingClassifier(
        estimators=[('lgbm', lgb_model), ('rf', rf_model)],
        voting='soft',
        n_jobs=-1,
    )


def main():
    # --- Phase 1: Ingest raw data and engineer kinematic features ---
    log_phase("PHASE 1: DATA INGESTION & KINEMATIC FEATURE ENGINEERING")
    train_df, test_df, feature_cols = load_data()

    logging.info("Applying kinematic feature engineering...")
    X_train = add_kinematic_features(train_df, feature_cols)
    X_test = add_kinematic_features(test_df, feature_cols)
    y_train = train_df['ActivityClass']

    logging.info(f"Engineered Train shape: {X_train.shape} | Test shape: {X_test.shape}")

    # --- Phase 2: Cross-validate to estimate generalisation performance ---
    log_phase("PHASE 2: MODEL SELECTION & VALIDATION")
    model = build_model()

    logging.info("Running 5-Fold Cross-Validation...")
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    logging.info(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    # --- Phase 3: Retrain on full data and generate submission file ---
    log_phase("PHASE 3: FULL TRAINING & SUBMISSION")
    logging.info("Training model on full dataset...")
    model.fit(X_train, y_train)

    logging.info("Generating predictions...")
    preds = model.predict(X_test)

    submission = pd.DataFrame({
        "Id": test_df["Id"],
        "ActivityClass": preds,
    })

    submission.to_csv(OUTPUT_PATH, index=False)
    logging.info(f"Saved {len(submission)} predictions to '{OUTPUT_PATH}'.")


if __name__ == "__main__":
    main()
